In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

In [2]:

df = pd.read_csv('data/pima-indians-diabetes.data', skiprows=2, header=None)

X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values
y = np.where(y == 0, -1, 1)   # tanh output uses -1 / +1


In [3]:

def tanh(x):
    x = np.clip(x, -500, 500)
    return np.tanh(x)

def tanh_derivative(output):
    # derivative of tanh using output value
    return 1 - output**2


In [4]:

def train_nn(
    X_train,
    y_train,
    lr=0.1,
    hidden_size=1,
    output_size=1,
    max_epochs=5000,
    target_success=0.85,
    wait_epoch=300,
    min_improvement=0.0,
    seed=42
):
    np.random.seed(seed)

    input_size = X_train.shape[1]

    W1 = np.random.randn(input_size, hidden_size) * 0.01
    W2 = np.random.randn(hidden_size + 1, output_size) * 0.01

    all_loss = []
    all_accuracy = []

    best_success = 0
    epochs_without_improvement = 0

    for epoch in range(max_epochs):
        epoch_loss = 0
        correct = 0

        for i in range(len(X_train)):
            x_i = X_train[i].reshape(1, -1)
            y_i = np.array([[y_train[i]]])

            # forward pass
            hidden_linear = x_i @ W1
            hidden_output = tanh(hidden_linear)
            hidden_output_bias = np.hstack((np.ones((hidden_output.shape[0], 1)), hidden_output))

            output_linear = hidden_output_bias @ W2
            predicted_output = tanh(output_linear)

            # loss
            sample_loss = np.mean((predicted_output - y_i) ** 2)
            epoch_loss += sample_loss

            # classification accuracy
            predicted_label = 1 if predicted_output.item() >= 0 else -1
            if predicted_label == y_i.item():
                correct += 1

            # backward pass
            delta_output = (predicted_output - y_i) * tanh_derivative(predicted_output)

            delta_hidden_full = (delta_output @ W2.T) * tanh_derivative(hidden_output_bias)
            delta_hidden = delta_hidden_full[:, 1:]   # remove bias column

            # weights update
            W2 = W2 - lr * (hidden_output_bias.T @ delta_output)
            W1 = W1 - lr * (x_i.T @ delta_hidden)

        current_loss = epoch_loss / len(X_train)
        success_rate = correct / len(X_train)

        all_loss.append(current_loss)
        all_accuracy.append(success_rate)

        # early stopping logic
        if success_rate > best_success + min_improvement:
            best_success = success_rate
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if best_success >= target_success and epochs_without_improvement >= wait_epoch:
            break

    return W1, W2, all_loss, all_accuracy

In [5]:

def evaluate_nn(X_data, y_data, W1, W2):
    correct = 0

    for i in range(len(X_data)):
        x_i = X_data[i].reshape(1, -1)

        hidden_linear = x_i @ W1
        hidden_output = tanh(hidden_linear)
        hidden_output_bias = np.hstack((np.ones((hidden_output.shape[0], 1)), hidden_output))

        output_linear = hidden_output_bias @ W2
        predicted_output = tanh(output_linear)

        predicted_label = 1 if predicted_output.item() >= 0 else -1

        if predicted_label == y_data[i]:
            correct += 1

    return correct / len(X_data)


In [6]:

kf = KFold(n_splits=10, shuffle=True, random_state=42)

fold_accuracies = []
fold_no = 1

for train_index, val_index in kf.split(X):
    X_train_raw, X_val_raw = X[train_index], X[val_index]
    y_train, y_val = y[train_index], y[val_index]

    # scale INSIDE each fold to avoid data leakage
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw)
    X_val = scaler.transform(X_val_raw)

    # add bias column
    X_train = np.hstack((np.ones((X_train.shape[0], 1)), X_train))
    X_val = np.hstack((np.ones((X_val.shape[0], 1)), X_val))

    # train fresh model for this fold
    W1, W2, loss_history, acc_history = train_nn(
        X_train,
        y_train,
        lr=0.1,
        hidden_size=1,
        output_size=1,
        max_epochs=5000,
        target_success=0.85,
        wait_epoch=300,
        min_improvement=0.0,
        seed=42 + fold_no   # slightly different init per fold
    )

    # validate on held-out fold
    val_accuracy = evaluate_nn(X_val, y_val, W1, W2)
    fold_accuracies.append(val_accuracy)

    print(f"Fold {fold_no}: Validation Accuracy = {val_accuracy:.4f}")
    fold_no += 1


Fold 1: Validation Accuracy = 0.7403
Fold 2: Validation Accuracy = 0.7532
Fold 3: Validation Accuracy = 0.6883
Fold 4: Validation Accuracy = 0.7143
Fold 5: Validation Accuracy = 0.7662
Fold 6: Validation Accuracy = 0.7013
Fold 7: Validation Accuracy = 0.7662
Fold 8: Validation Accuracy = 0.7273
Fold 9: Validation Accuracy = 0.6974
Fold 10: Validation Accuracy = 0.7368


In [7]:

mean_accuracy = np.mean(fold_accuracies)
std_accuracy = np.std(fold_accuracies)

print("\n10-Fold Cross Validation Results")
print("Fold Accuracies:", [f"{acc:.4f}" for acc in fold_accuracies])
print(f"Average Validation Accuracy: {mean_accuracy:.4f}")
print(f"Standard Deviation: {std_accuracy:.4f}")


10-Fold Cross Validation Results
Fold Accuracies: ['0.7403', '0.7532', '0.6883', '0.7143', '0.7662', '0.7013', '0.7662', '0.7273', '0.6974', '0.7368']
Average Validation Accuracy: 0.7291
Standard Deviation: 0.0268
